# Inspección inicial del dataset DEAP en formato RAW `.bdf`

## Objetivo

El objetivo de este notebook es inspeccionar la estructura del dataset DEAP descargado en formato `.bdf`.

Queremos responder:

1. ¿Cuántos archivos `.bdf` hay?
2. ¿Corresponden a los 32 sujetos esperados?
3. ¿Cada archivo puede abrirse correctamente?
4. ¿Qué canales contiene cada archivo?
5. ¿Cuál es la frecuencia de muestreo?
6. ¿Cuál es la duración aproximada por sujeto?
7. ¿El tamaño descargado parece compatible con el dataset RAW completo?

In [1]:
from pathlib import Path
from typing import List

DATASET_DIR: Path = Path("dataset")

bdf_files: List[Path] = sorted(DATASET_DIR.glob("*.bdf"))

print(f"Carpeta del dataset: {DATASET_DIR.resolve()}")
print(f"Número de archivos .bdf encontrados: {len(bdf_files)}")

for file_path in bdf_files:
    print(file_path.name)

Carpeta del dataset: /home/russell/ssd/code/Topicos_Ciencia_Datos/Visual_Analytic_DEAP/dataset
Número de archivos .bdf encontrados: 32
s01.bdf
s02.bdf
s03.bdf
s04.bdf
s05.bdf
s06.bdf
s07.bdf
s08.bdf
s09.bdf
s10.bdf
s11.bdf
s12.bdf
s13.bdf
s14.bdf
s15.bdf
s16.bdf
s17.bdf
s18.bdf
s19.bdf
s20.bdf
s21.bdf
s22.bdf
s23.bdf
s24.bdf
s25.bdf
s26.bdf
s27.bdf
s28.bdf
s29.bdf
s30.bdf
s31.bdf
s32.bdf


## Verificación inicial

DEAP RAW debería contener un archivo `.bdf` por sujeto.

En teoría:

- `s01.bdf`
- `s02.bdf`
- ...
- `s32.bdf`

Por lo tanto, esperamos encontrar **32 archivos `.bdf`**.

## Tamaño total descargado

Ahora calculamos cuánto pesan los archivos `.bdf`.

Esto nos ayuda a verificar si el dataset parece completo.  
Un conjunto RAW con 32 archivos `.bdf` puede pesar varios GB.

In [2]:
from typing import Tuple


def bytes_to_gb(size_bytes: int) -> float:
    return size_bytes / (1024**3)


file_sizes: List[Tuple[str, float]] = [
    (file_path.name, bytes_to_gb(file_path.stat().st_size)) for file_path in bdf_files
]

total_size_gb: float = sum(size_gb for _, size_gb in file_sizes)

print(f"Tamaño total aproximado: {total_size_gb:.2f} GB")

for file_name, size_gb in file_sizes:
    print(f"{file_name}: {size_gb:.2f} GB")

Tamaño total aproximado: 8.25 GB
s01.bdf: 0.27 GB
s02.bdf: 0.25 GB
s03.bdf: 0.27 GB
s04.bdf: 0.23 GB
s05.bdf: 0.27 GB
s06.bdf: 0.25 GB
s07.bdf: 0.25 GB
s08.bdf: 0.24 GB
s09.bdf: 0.26 GB
s10.bdf: 0.24 GB
s11.bdf: 0.29 GB
s12.bdf: 0.24 GB
s13.bdf: 0.25 GB
s14.bdf: 0.27 GB
s15.bdf: 0.28 GB
s16.bdf: 0.25 GB
s17.bdf: 0.24 GB
s18.bdf: 0.25 GB
s19.bdf: 0.26 GB
s20.bdf: 0.24 GB
s21.bdf: 0.24 GB
s22.bdf: 0.26 GB
s23.bdf: 0.25 GB
s24.bdf: 0.32 GB
s25.bdf: 0.28 GB
s26.bdf: 0.27 GB
s27.bdf: 0.26 GB
s28.bdf: 0.24 GB
s29.bdf: 0.28 GB
s30.bdf: 0.26 GB
s31.bdf: 0.26 GB
s32.bdf: 0.25 GB


## Lectura de un archivo `.bdf`

Ahora abriremos un solo archivo, empezando con `s01.bdf`.

No vamos a procesar todo el dataset todavía.  
Primero necesitamos entender la estructura de un sujeto.

In [3]:
import mne
from mne.io import BaseRaw

sample_file: Path = DATASET_DIR / "s01.bdf"

raw: BaseRaw = mne.io.read_raw_bdf(input_fname=sample_file, preload=False, verbose=True)

print(raw)

Extracting BDF parameters from dataset/s01.bdf...
Setting channel info structure...
Creating raw.info structure...
<RawBDF | s01.bdf, 48 x 1980928 (3869.0 s), ~50 KiB, data not loaded>


## Información general del archivo

Aquí inspeccionamos:

- número de canales
- frecuencia de muestreo
- duración total
- nombres de canales

In [4]:
from typing import Any, Dict

info: Dict[str, Any] = raw.info

sampling_frequency: float = float(info["sfreq"])
channel_names: List[str] = raw.ch_names
num_channels: int = len(channel_names)
duration_seconds: float = raw.n_times / sampling_frequency
duration_minutes: float = duration_seconds / 60.0

print(f"Número de canales: {num_channels}")
print(f"Frecuencia de muestreo: {sampling_frequency} Hz")
print(f"Número de muestras: {raw.n_times}")
print(f"Duración aproximada: {duration_minutes:.2f} minutos")

print("\nCanales:")
for index, channel_name in enumerate(channel_names, start=1):
    print(f"{index:02d}. {channel_name}")

Número de canales: 48
Frecuencia de muestreo: 512.0 Hz
Número de muestras: 1980928
Duración aproximada: 64.48 minutos

Canales:
01. Fp1
02. AF3
03. F7
04. F3
05. FC1
06. FC5
07. T7
08. C3
09. CP1
10. CP5
11. P7
12. P3
13. Pz
14. PO3
15. O1
16. Oz
17. O2
18. PO4
19. P4
20. P8
21. CP6
22. CP2
23. C4
24. T8
25. FC6
26. FC2
27. F4
28. F8
29. AF4
30. Fp2
31. Fz
32. Cz
33. EXG1
34. EXG2
35. EXG3
36. EXG4
37. EXG5
38. EXG6
39. EXG7
40. EXG8
41. GSR1
42. GSR2
43. Erg1
44. Erg2
45. Resp
46. Plet
47. Temp
48. Status


## Detección de eventos (trials)

En el dataset DEAP, los eventos que marcan el inicio y fin de los trials están codificados en el canal "Status".

El objetivo de esta sección es:

- extraer los eventos
- identificar cuántos trials hay
- verificar si realmente hay 40 trials

In [5]:
from typing import Optional
import numpy as np

# MNE detecta eventos desde el canal Status
events: np.ndarray = mne.find_events(raw, stim_channel="Status", shortest_event=1)

print(f"Número total de eventos detectados: {len(events)}")

print("\nPrimeros eventos:")
print(events[:10])

Finding events on: Status
Trigger channel Status has a non-zero initial value of 65536 (consider using initial_event=True to detect this event)
Removing orphaned offset at the beginning of the file.
13760 events found on stim channel Status
Event IDs: [1 2 3 4 5 6 7]
Número total de eventos detectados: 13760

Primeros eventos:
[[  1960      0      1]
 [  1972      0      1]
 [ 63424      0      2]
 [ 65045      0      3]
 [ 67621      0      4]
 [ 98407      0      5]
 [108034      0      1]
 [112892      0      1]
 [115092      0      1]
 [116099      0      1]]


## Análisis de códigos de eventos

El canal `Status` contiene muchos eventos internos.  
Antes de separar trials, necesitamos saber cuántas veces aparece cada código y en qué momentos aparecen.

El objetivo es identificar qué código corresponde al inicio de cada trial o segmento experimental.

In [6]:
from typing import Dict
import numpy as np

event_codes: np.ndarray = events[:, 2]

event_counts: Dict[int, int] = {
    int(code): int(np.sum(event_codes == code)) for code in np.unique(event_codes)
}

print("Conteo de eventos por código:")
for code, count in event_counts.items():
    print(f"Código {code}: {count} eventos")

print("\nPrimeros 30 eventos:")
print(events[:30])

Conteo de eventos por código:
Código 1: 162 eventos
Código 2: 2 eventos
Código 3: 40 eventos
Código 4: 40 eventos
Código 5: 40 eventos
Código 6: 13475 eventos
Código 7: 1 eventos

Primeros 30 eventos:
[[  1960      0      1]
 [  1972      0      1]
 [ 63424      0      2]
 [ 65045      0      3]
 [ 67621      0      4]
 [ 98407      0      5]
 [108034      0      1]
 [112892      0      1]
 [115092      0      1]
 [116099      0      1]
 [117216      0      3]
 [119792      0      4]
 [150544      0      5]
 [155342      0      1]
 [157418      0      1]
 [158700      0      1]
 [160748      0      1]
 [161855      0      3]
 [164431      0      4]
 [195183      0      5]
 [199773      0      1]
 [203877      0      1]
 [204775      0      1]
 [207199      0      1]
 [208294      0      3]
 [210870      0      4]
 [241622      0      5]
 [244472      0      1]
 [245885      0      1]
 [247113      0      1]]


In [7]:
import numpy as np
from typing import Tuple

# obtener datos (solo una pequeña parte)
data: np.ndarray = raw.get_data(start=0, stop=10)

print("Shape:", data.shape)

print("\nPrimeros valores (primeros 5 canales):")
print(data[:5, :10])

Shape: (48, 10)

Primeros valores (primeros 5 canales):
[[ 0.0059471   0.00595241  0.00597335  0.00598838  0.00598735  0.00597894
   0.00598082  0.00597019  0.00596016  0.00595494]
 [ 0.00234111  0.00233998  0.00233051  0.00235114  0.00237457  0.00237482
   0.00238204  0.00237107  0.00235507  0.00235164]
 [ 0.00114611  0.00115104  0.00116701  0.00118195  0.00118136  0.00116811
   0.00117001  0.00117108  0.00115836  0.00115536]
 [-0.00273017 -0.00273104 -0.00271429 -0.00270789 -0.00271651 -0.00272085
  -0.00271782 -0.00272439 -0.00273332 -0.00273557]
 [ 0.01341415  0.01341137  0.01342452  0.01343518  0.01343312  0.0134308
   0.01343458  0.01342368  0.01341052  0.01341062]]
